# `lga` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'lga'
feature_metadata = {'order': 15, 'name': 'lga', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain with explicit unseen handling', 'finding': 'All 125 training levels are covered in test and each maps to one region.', 'decision': 'Retain and compare with the coarser region representation.', 'risk': 'Strong geographic target differences require grouped robustness checks.', 'related': [{'feature': 'region', 'reason': 'LGA maps deterministically to region in the supplied data.'}, {'feature': 'ward', 'reason': 'LGA context disambiguates reused ward names.'}, {'feature': 'region_code', 'reason': 'Code relationships expose small administrative anomalies.'}, {'feature': 'district_code', 'reason': 'Named and coded district representations overlap.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for lga.


## Supported target evidence


In [2]:
sentinel_tokens = []
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
lga,,,,,
njombe,2503,True,80.18,3.76,16.06
arusha rural,1252,True,69.89,3.83,26.28
moshi rural,1251,True,58.59,9.51,31.89
bariadi,1177,True,49.28,34.75,15.97
rungwe,1106,True,61.12,14.56,24.32
kilosa,1094,True,53.66,6.67,39.67
kasulu,1047,True,58.36,19.20,22.45
mbozi,1034,True,43.52,6.77,49.71
meru,1009,True,65.11,3.17,31.71


status_group,rows,non functional (%)
lga,,
nachingwea,300,86.33
pangani,305,77.70
ruangwa,291,75.26
nanyumbu,158,73.42
rorya,210,73.33
liwale,154,72.73
sikonge,170,71.18
bunda,438,68.26
tandahimba,266,68.05


## Observation

All 125 training levels are covered in test and each maps to one region.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Retain and compare with the coarser region representation.

**Risk to carry forward:** Strong geographic target differences require grouped robustness checks.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
lga,candidate,retain with explicit unseen handling,All 125 training levels are covered in test an...,Retain and compare with the coarser region rep...,Strong geographic target differences require g...
